# Experiment 1 — a skewed join, baseline

`orders_skew` (14 M rows) joined with `customers_big` (2 M rows, ~149 MiB) on `customer_id`. Full computation into the `noop` sink.
AQE's skew handling is switched off for this baseline so that the raw effect is visible; Experiment 3 turns it back on.
Spark UI: http://localhost:4040 → **Stages** (Summary Metrics; Tasks table sorted by duration), **SQL / DataFrame**.

In [1]:
import time
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("m5-exp1-baseline").getOrCreate()
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")

fact = spark.read.parquet("/opt/data/m5/orders_skew")
dim = spark.read.parquet("/opt/data/m4/customers_big")

def timed(label, df):
    t0 = time.perf_counter()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label}: {time.perf_counter() - t0:.1f}s")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 18:30:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

## How skewed is the key?

In [2]:
fact.groupBy("customer_id").count().orderBy(F.desc("count")).show(5)

[Stage 2:=============================>                             (4 + 4) / 8]

+-----------+-------+
|customer_id|  count|
+-----------+-------+
|          1|6000000|
|       9577|    908|
|       9104|    894|
|       5181|    894|
|       5780|    893|
+-----------+-------+
only showing top 5 rows


## The join

**PREDICT:** the join strategy; which stage will show the skew; what its task *duration* and *shuffle read* summary (min / median / max) will look like; whether spill appears; the runtime.

In [3]:
joined = fact.join(dim, "customer_id")
joined.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- SortMergeJoin Inner (9)
      :- Sort (4)
      :  +- Exchange (3)
      :     +- Filter (2)
      :        +- Scan parquet  (1)
      +- Sort (8)
         +- Exchange (7)
            +- Filter (6)
               +- Scan parquet  (5)


(1) Scan parquet 
Output [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m5/orders_skew]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date,note:string>

(2) Filter
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Condition : isnotnull(customer_id#1)

(3) Exchange
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Arguments: hashpartitioning(customer_id#1, 200), ENSURE_REQUIREMENTS, [plan_id=78]

(4) Sort
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amou

In [4]:
timed("baseline skewed join", joined)

[Stage 9:====================================================>    (11 + 1) / 12]

baseline skewed join: 4.4s


In [5]:
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")
timed("baseline, no coalescing", joined)

[Stage 14:=====================================================>(199 + 1) / 200]

baseline, no coalescing: 3.7s


26/09/30 19:22:27 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/blockmgr-4b5a40d0-8ca6-4db1-8516-b8d4613e5cf8. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/blockmgr-4b5a40d0-8ca6-4db1-8516-b8d4613e5cf8
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:352)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:269)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:248)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:158)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:157)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1062)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1(DiskBlockManager.scala:372)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1$adapted(DiskBlockManager.scala:368)
	at scala.collection.ArrayOps$.foreach

## Observe

Stages tab → the join stage → Summary Metrics (Duration, Shuffle Read Size / Records, Spill) and the Tasks table sorted by Duration descending.
SQL / DataFrame tab → the query → the join node's metrics.

## Stop before opening the next notebook

In [ ]:
spark.stop()